## Load RCSB PDB data and settings

In [2]:
# Load the full RCSB protein structure dataset from a JSON file.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()  # repository root (the notebooks live here)
# Inputs: example_data/ ships with the repo; point DATA_DIR at your own data folder for full-scale runs.
DATA_DIR = PROJECT_ROOT / "example_data"
ANNOTATION_DIR = DATA_DIR / "Official_annotation"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "question_generation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RCSB_DATA_PATH = DATA_DIR / "full_rcsb_data.json"

with open(RCSB_DATA_PATH, "r") as f:
    data = json.load(f)


In [3]:
# Import ML/API libraries and configure the LLM backend (Qwen, LLaMA, or Gemini) for question generation.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, json
import google.generativeai as genai



model_type = 'qwen'#'qwen'

# abstracts = abstracts[0]
# if len(abstracts) == 1:
#     generation_type = 1
# else:
#     generation_type = 2
    
# print(f'generation_type: {generation_type}')


if 'gemini' in model_type:
    model_name = 'gemini-2.0-flash-lite'#"Qwen/Qwen2.5-7B-Instruct"
    google_api_key = 'your_api_key'
else:
    if model_type == 'qwen':
        model_name = "Qwen/Qwen2.5-7B-Instruct"
    elif model_type == 'llama':
        model_name = "meta-llama/Llama-3.1-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,   # safer than float16 if GPU supports
        device_map={"": 1}   #"auto"
    )
    


/home/kding1/anaconda3/envs/RAD_AI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.87it/s]


In [4]:
# Ensure the tokenizer has a pad token, defaulting to the EOS token if missing.
# Ensure pad token is defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [5]:
# Check the total number of protein entries in the dataset.
len(data['data']['entries'])

5

In [6]:
# Count how many entries have PubMed information available.
from tqdm import tqdm
c = 0
total = 0
for entry in tqdm(data['data']['entries']):
    pubmed_info = entry['pubmed']
    rcsb_id = entry['rcsb_id']
    if pubmed_info != None:
        c += 1
        # if rcsb_id == '6M4F':#'6M55':
        #     print(rcsb_id)
    total += 1
print(c, total)

100%|██████████| 5/5 [00:00<00:00, 57456.22it/s]

5 5


In [7]:
# Inspect all metadata for a single sample entry (index 5).
# i = 5


# pubmed_category = data['data']['entries'][i]

# pubmed_category

In [8]:
# Build core lookup dictionaries: RCSB ID → abstract, category counts, and RCSB ID → PubMed ID.
from collections import defaultdict

category = defaultdict(list)
counts = defaultdict(int)
dict_rcsb_id_abstract = {}
scientific_name = defaultdict(list)
dict_rid_pid = {}


dict_rid_info = {}
c = len(data['data']['entries'])
for i in tqdm(range(c)):

    try:
        pubmed_info = data['data']['entries'][i]['pubmed']
    except:
        continue

        
    rcsb_id = data['data']['entries'][i]['rcsb_id']
    try:
        dict_rcsb_id_abstract[rcsb_id] = data['data']['entries'][i]['pubmed']['rcsb_pubmed_abstract_text']
    except:
        continue

    pubmed_id = data['data']['entries'][i]['pubmed']['rcsb_pubmed_container_identifiers']['pubmed_id']
    rcsb_id = data['data']['entries'][i]['rcsb_id']
    
    dict_rid_pid[rcsb_id] = pubmed_id
    
    try:
        pubmed_category = data['data']['entries'][i]['pubmed']['rcsb_pubmed_mesh_descriptors']
    
        for pub_c in pubmed_category:
            category[pub_c].append(rcsb_id)
            # counts[pub_c] += 1
    except:
        category['No category'].append(rcsb_id)
    
    try:
        rcsb_categories = list(set([item['rcsb_entity_source_organism'][-1]['ncbi_scientific_name'] for item in data['data']['entries'][i]['polymer_entities']]))
    
        for rcsb_category in rcsb_categories:
            scientific_name[rcsb_category].append(rcsb_id)
            counts[rcsb_category] += 1
    except:
        scientific_name['No category'].append(rcsb_id)
        counts['No category'] += 1

    dict_rid_info[rcsb_id] = {}
    dict_rid_info[rcsb_id]['pubmed_id'] = pubmed_id
    dict_rid_info[rcsb_id]['rcsb_pubmed_doi'] = data['data']['entries'][i]['pubmed']['rcsb_pubmed_doi']
    dict_rid_info[rcsb_id]['abstract'] = data['data']['entries'][i]['pubmed']['rcsb_pubmed_abstract_text']
    dict_rid_info[rcsb_id]['pubmed_title'] = data['data']['entries'][i]['citation'][0]['title']
    dict_rid_info[rcsb_id]['pdb_title'] = data['data']['entries'][i]['struct']['title']

# dict_rcsb_id_abstract = {}
# for rcsb_id in dict_rid_info.keys():
#     dict_rcsb_id_abstract[rcsb_id] = dict_rid_info[rcsb_id]['abstract']
    

    
dict_rcsb_id_abstract = {}
results = []
for rcsb_id in dict_rid_info.keys():
    dict_pid_info = {}
    dict_rcsb_id_abstract[rcsb_id] = dict_rid_info[rcsb_id]['abstract']
     
    if dict_rid_info[rcsb_id]['abstract'] != None:
        dict_pid_info['PubMed_ID'] = dict_rid_info[rcsb_id]['pubmed_id']
        dict_pid_info['Abstract'] = dict_rid_info[rcsb_id]['abstract']
        results.append(dict_pid_info)  

    




100%|██████████| 5/5 [00:00<00:00, 26886.56it/s]


In [9]:
# Verify info and abstract content for one RCSB ID.
example_rid = next(iter(dict_rid_info))
print(dict_rid_info[example_rid])
print()
print(dict_rcsb_id_abstract[example_rid])

{'pubmed_id': 28396445, 'rcsb_pubmed_doi': '10.1073/pnas.1700248114', 'abstract': 'Cyclic nucleotide-gated (CNG) and hyperpolarization-activated cyclic nucleotide-regulated (HCN) ion channels play crucial physiological roles in phototransduction, olfaction, and cardiac pace making. These channels are characterized by the presence of a carboxyl-terminal cyclic nucleotide-binding domain (CNBD) that connects to the channel pore via a C-linker domain. Although cyclic nucleotide binding has been shown to promote CNG and HCN channel opening, the precise mechanism underlying gating remains poorly understood. Here we used cryoEM to determine the structure of the intact LliK CNG channel isolated from  <i>Leptospira licerasiae </i>-which shares sequence similarity to eukaryotic CNG and HCN channels-in the presence of a saturating concentration of cAMP. A short S4-S5 linker connects nearby voltage-sensing and pore domains to produce a non-domain-swapped transmembrane architecture, which appears t

In [10]:
# Build reverse mapping from PubMed ID to a list of associated RCSB IDs.
from collections import defaultdict

dict_pid_rid = defaultdict(list)

for rid, pid in dict_rid_pid.items():
    dict_pid_rid[pid].append(rid)


In [11]:
# (Debug) Look up RCSB IDs linked to a specific PubMed ID.
# dict_pid_rid[29596881]

## data exploration

In [12]:
# Sort the category frequency dictionary in descending order.
sorted_dict = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True))
sorted_dict

{'Leptospira licerasiae serovar Varillal str. VAR 010': 1,
 'Mesorhizobium sp. LUK': 1,
 'Mus musculus': 1,
 'Homo sapiens': 1,
 'Respirovirus muris': 1,
 'Secale cereale': 1,
 'Talaromyces marneffei': 1}

In [13]:
# Find categories that appear in more than 200 entries.
keys = [k for k, v in sorted_dict.items() if v > 200]
print(len(keys))

0


In [14]:
# Print the count of categories for each frequency range (1–10 entries each).
left = 1
right = 10
for i in range(1,11):
    keys = [k for k, v in sorted_dict.items() if left <= v <= right]
    print(f'In range [{left}:{right}]: {len(keys)}')
    left += 10
    right +=10
keys = [k for k, v in sorted_dict.items() if v>100]
print(f'In range [100:]: {len(keys)}')

In range [1:10]: 7
In range [11:20]: 0
In range [21:30]: 0
In range [31:40]: 0
In range [41:50]: 0
In range [51:60]: 0
In range [61:70]: 0
In range [71:80]: 0
In range [81:90]: 0
In range [91:100]: 0
In range [100:]: 0


In [15]:
# Collect categories with 1–30 entries (candidates for single-abstract mode).
keys = [k for k, v in sorted_dict.items() if 1 <= v <= 30]
print(len(keys))

7


In [16]:
# Collect categories with more than 30 entries (candidates for multi-abstract mode).
keys = [k for k, v in sorted_dict.items() if v>30]
print(len(keys))

0


In [17]:
# Select and randomly sample categories based on single vs. multiple abstract mode.
import random 

selected = 'single'
if selected == 'single':
    k = 1
    sorted_category = [key for key, v in sorted_dict.items() if 1 <= v <= 30]
    print(len(sorted_category))
    sorted_category = random.sample(sorted_category, min(400, len(sorted_category))) 
    print(len(sorted_category))
else:
    k = 2
    sorted_category = [key for key, v in sorted_dict.items() if v>30]
    print(len(sorted_category))
    sorted_category = random.sample(sorted_category, min(200, len(sorted_category))) 
    print(len(sorted_category))

7
7


In [18]:
# Display the sampled category list.
sorted_category

['Homo sapiens',
 'Leptospira licerasiae serovar Varillal str. VAR 010',
 'Secale cereale',
 'Talaromyces marneffei',
 'Mesorhizobium sp. LUK',
 'Mus musculus',
 'Respirovirus muris']

In [19]:
# Print the number of selected categories.
print(len(sorted_category))

7


## Select category by avoiding using previously used categories

In [20]:
# Load previously used category/PubMed/PDB tracking info from previous annotation runs to avoid repeated category usage.
import json

R1_path = ANNOTATION_DIR / "Official_annotation_R1" / "used_info_single_multiple.json"
with open(R1_path, "r", encoding="utf-8") as f:
    R1_used_info = json.load(f)



R2_path = ANNOTATION_DIR / "Official_annotation_R2" / "used_info_single.json"
with open(R2_path, "r", encoding="utf-8") as f:
    R2_used_info = json.load(f)

R2_path = ANNOTATION_DIR / "Official_annotation_R2" / "used_info_multiple.json"
with open(R2_path, "r", encoding="utf-8") as f:
    R2_used_info_multiple = json.load(f)



R3_path = ANNOTATION_DIR / "Official_annotation_R3" / "used_info_single.json"
with open(R3_path, "r", encoding="utf-8") as f:
    R3_used_info = json.load(f)

R3_path = ANNOTATION_DIR / "Official_annotation_R3" / "used_info_multiple.json"
with open(R3_path, "r", encoding="utf-8") as f:
    R3_used_info_multiple = json.load(f)

In [21]:
# Inspect available keys in the R3 used-info dictionary.
R3_used_info.keys()

dict_keys(['used_category_name_single', 'used_pid_single', 'used_pdb_id_single'])

In [22]:
# Merge single and multiple category/PID lists from the R1 annotation to exclude already-used data.
used = R1_used_info['used_category_name_single'] + R1_used_info['used_category_name_multiple']
used_pid = R1_used_info['used_pid_single'] + R1_used_info['used_pid_multiple']
used_pdb = R1_used_info['used_pdb_id_single'] + R1_used_info['used_pdb_id_multiple']

print(len(used))
print(len(used_pid))
print(len(used_pdb))

used += R2_used_info['used_category_name_single']
used_pid += R2_used_info['used_pid_single']
used_pdb += R2_used_info['used_pdb_id_single']


print(len(used))
print(len(used_pid))
print(len(used_pdb))

used += R2_used_info_multiple['used_category_name_multiple']
used_pid += R2_used_info_multiple['used_pid_multiple']
used_pdb += R2_used_info_multiple['used_pdb_id_multiple']


print(len(used))
print(len(used_pid))
print(len(used_pdb))

used += R3_used_info['used_category_name_single']
used_pid += R3_used_info['used_pid_single']
used_pdb += R3_used_info['used_pdb_id_single']


print(len(used))
print(len(used_pid))
print(len(used_pdb))

used += R3_used_info_multiple['used_category_name_multiple']
used_pid += R3_used_info_multiple['used_pid_multiple']
used_pdb += R3_used_info_multiple['used_pdb_id_multiple']


print(len(used))
print(len(used_pid))
print(len(used_pdb))

4
4
4
6
6
6
8
8
8
10
10
10
12
12
12


In [23]:
# Compute entry count statistics for already-used categories.
count = []
no = 0
for i in used:
    if i in sorted_dict.keys():
        count.append(sorted_dict[i])
    else:
        no += 1
max(count, default=0)



0

In [24]:
# Select candidate RCSB IDs from the chosen categories, excluding already-used ones.
import random


category_query_name ='all' #'Zika virus'#'immunodeficiency virus'#'all'#'HIV'
# category_query_name_list =[ 'Danio rerio', 'Yersinia pestis', 'Mycobacterium tuberculosis', 'Zika virus', 'HIV', 'Influenza A virus', 'Toxoplasma gondii', 'Staphylococcus aureus', 'Streptococcus pneumoniae']

# category_query_name2 = 'immunodeficiency virus'#'all'#'HIV'
candidates = []

# for i in scientific_name.keys():
for i in sorted_category:
    if category_query_name == 'all':
        if i not in used:
            cur_candidates = scientific_name[i]
            cur_candidates = random.sample(cur_candidates, k)
            valid = []
            for j in cur_candidates:
                if dict_rid_pid[j] not in used_pid:
                    valid.append(j)
                else:
                    print(j)
            candidates += valid
        else:
            print(i)
    else:
        try:
            if i in category_query_name_list:
                candidates += scientific_name[i]
        except:
            continue
a = candidates

In [25]:
# Build abstract-to-RCSB-ID and abstract-to-PubMed-ID lookup dicts from candidates.
abstracts = []
dict_abs_rid = {}
dict_abs_pid = {}
for i in candidates:
    dict_abs_rid[dict_rcsb_id_abstract[i]] = i
    dict_abs_pid[dict_rcsb_id_abstract[i]] = dict_rid_pid[i]
    print(dict_rcsb_id_abstract[i])
    print()
    if dict_rcsb_id_abstract[i] != None:
        abstracts.append(dict_rcsb_id_abstract[i])

Two synthetic O-GlcNAc-bearing peptides that elicit H-2Db-restricted glycopeptide-specific cytotoxic T cells (CTL) have been shown to display nonreciprocal patterns of cross-reactivity. Here, we present the crystal structures of the H-2Db glycopeptide complexes to 2.85 A resolution or better. In both cases, the glycan is solvent exposed and available for direct recognition by the T cell receptor (TCR). We have modeled the complex formed between the MHC-glycopeptide complexes and their respective TCRs, showing that a single saccharide residue can be accommodated in the standard TCR-MHC geometry. The models also reveal a possible molecular basis for the observed cross-reactivity patterns of the CTL clones, which appear to be influenced by the length of the CDR3 loop and the nature of the immunizing ligand.

Cyclic nucleotide-gated (CNG) and hyperpolarization-activated cyclic nucleotide-regulated (HCN) ion channels play crucial physiological roles in phototransduction, olfaction, and card

In [26]:
# Deduplicate the abstracts list.
abstracts = list(set(abstracts))

# print(abstracts)
print(len(abstracts))

5


## Single-abstract question generation

In [27]:
# Prepare abstract dicts and prompt message batches for LLM question generation.
generation_type = 2


# Abstract pairs
pairs = []
pid_order = []
dict_abstracts = {}
for i,abstract in enumerate(abstracts):
    pid_order.append(dict_abs_pid[abstract])
    dict_abstracts[f"Abstract {i+1}"] = abstract

task_prompt = """
I have an abstract describing the structure and function of a protein or protein complex. I want you to generate a list of high-level, biologically meaningful questions that a PDB user might ask an AI search tool on RCSB.org to find relevant structures. The questions should:
    1. Be focused on searching for structures in the PDB.
    2. Be phrased like “Can you find X” or “Can you give me structures of Y” — concise, clear, and actionable.
    3. Highlight biological relevance or functional aspects, not overly specific residue-level details.
    4. Be answerable based on the information in the abstract.
    5. Avoid unnecessary jargon; they should read like real search queries a scientist might enter.
    6. Either the full name or abbreviation. Do not let them occur at the same time.
For example, if the abstract is about hemagglutinin from influenza viruses, suitable questions could be:
  - Can you find structures of H3 hemagglutinin from avian influenza viruses?
  - Can you show me hemagglutinin structures bound to sialic acid–containing sugars?
  - Can you give me the structures of HA relevant to cross-species influenza transmission?

Response format:
Question: {generated question}
Reference: {a list of reference sentences}
"""


filter_prompt = """
From the list of search questions you just generated, select the ones that are most useful for a typical PDB user. Choose questions that:

  1. Are high- or mid-level (not overly specific or residue-level).
  2. Are biologically meaningful and focused on finding structures relevant to a biological or biochemical concept.
  3. Are broad enough that the search results would include the structure described in the abstract **as well as** other related structures from different studies.
  4. Are purely about **searching for structures**, and **do not** involve performing or suggesting any kind of comparison, interpretation, or analysis (e.g., avoid phrases like “for comparison,” “to analyze,” “to study,” etc.).

In response, only include one best question that meets these criteria and directly copy the list of reference sentences. If needed, lightly rephrase the question to make it clearer, more general, and more typical of what a PDB user would ask.

Response format:
Question: {the selected question}
Reference: {a list of reference sentences}
"""

# task_prompt = """
# Based on the abstract:
# - Question: Give me a question that PDB users might ask to retrieve specific structures related to a wide variety of biological relevance and functions. (E.g., “Can you find a protein with 2 domains connected by a flexible link” or “Can you give me structures (from homo sapiens) containing a disulfide bond”, etc.).
# - Answer: Give me an answer based on these abstracts for this question. The answer should include a clear response and a corresponding brief explanation, based solely on the abstract.
# - Reference: Give me a reference to the answer by copying sentences directly from abstract.

# Response format:
# Question: {generated question}
# Answer: {generated answer}
# Reference: {reference}
# """

if 'Qwen' in model_name or 'llama':
    # tokenizer = AutoTokenizer.from_pretrained(model_name)
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_name,
    #     torch_dtype=torch.bfloat16,   # safer than float16 if GPU supports
    #     device_map={"": 2}   #"auto"
    # )
    
    
    # Build chat messages batch
    messages_batch = []

    if generation_type == 2:
        for name, abstract in dict_abstracts.items():

            abstract_prompt = f'{name}: {abstract}'
            messages = [
                {"role": "system", "content": "You are a PDB user who might ask to retrieve specific structures related to a wide variety of biological relevance and functions. (E.g., “Can you find a protein with 2 domains connected by a flexible link” or “Can you give me structures (from homo sapiens) containing a disulfide bond”, etc.)."},
                {"role": "user", "content": f"""{task_prompt}\n\n Now, given the abstract below, generate five high-level search questions suitable for a PDB user based on the above instruction:{abstract_prompt}"""}
            ]
            
            # messages = [
            #     {"role": "system", "content": "You are a PDB user who might ask to retrieve specific structures related to a wide variety of biological relevance and functions. (E.g., “Can you find a protein with 2 domains connected by a flexible link” or “Can you give me structures (from homo sapiens) containing a disulfide bond”, etc.)."},
            #     {"role": "user", "content": f"""You are given a scientific abstract:{abstract_prompt}\n\n{task_prompt}"""}
            # ]
            messages_batch.append(messages)

    batch_size = 20
    results = []
    
    for start in tqdm(range(0, len(messages_batch), batch_size)):
        batch_messages_batch = messages_batch[start:start + batch_size]
        chat_prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in batch_messages_batch
        ]
    
    
        # Tokenize batch
        inputs = tokenizer(chat_prompts, return_tensors="pt", padding=True, truncation=True, max_length=4096).to(model.device)
        
        # Generate
        outputs = model.generate(**inputs, max_new_tokens=800, temperature=0.2, top_p=0.9, eos_token_id=tokenizer.eos_token_id)
        
        # Decode & parse JSON
        for out in outputs:
            text = tokenizer.decode(out, skip_special_tokens=True)
            try:
                parsed = json.loads(text[text.find("{"): text.rfind("}")+1])
            except Exception:
                parsed = {"raw_output": text}
            results.append(parsed)
        # break

    messages_batch2 = []
    # filtering  
    for i in range(len(results)):
        results[i]['raw_output'] = results[i]['raw_output'].split('assistant')[-1]
        results[i]['raw_output'] = results[i]['raw_output'].split('Assistant')[-1]
        
    for entry in results:
        questions = entry['raw_output']
        messages = [
            {"role": "system", "content": "You are good at selecting the most useful question based on the provided instruction for a typical PDB user"},
            {"role": "user", "content": f"""{filter_prompt}\n\n Now, given the generated questions below:{questions}"""}
        ]
        
        # messages = [
        #     {"role": "system", "content": "You are a PDB user who might ask to retrieve specific structures related to a wide variety of biological relevance and functions. (E.g., “Can you find a protein with 2 domains connected by a flexible link” or “Can you give me structures (from homo sapiens) containing a disulfide bond”, etc.)."},
        #     {"role": "user", "content": f"""You are given a scientific abstract:{abstract_prompt}\n\n{task_prompt}"""}
        # ]
        messages_batch2.append(messages)

    batch_size = 20
    results_final = []
    
    for start in tqdm(range(0, len(messages_batch2), batch_size)):
        batch_messages_batch = messages_batch2[start:start + batch_size]
        chat_prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in batch_messages_batch
        ]
    
    
        # Tokenize batch
        inputs = tokenizer(chat_prompts, return_tensors="pt", padding=True, truncation=True, max_length=4096).to(model.device)
        
        # Generate
        outputs = model.generate(**inputs, max_new_tokens=800, temperature=0.2, top_p=0.9, eos_token_id=tokenizer.eos_token_id)
        
        # Decode & parse JSON
        for out in outputs:
            text = tokenizer.decode(out, skip_special_tokens=True)
            try:
                parsed = json.loads(text[text.find("{"): text.rfind("}")+1])
            except Exception:
                parsed = {"raw_output": text}
            results_final.append(parsed)
        # break
        
    # Print nicely
    print(json.dumps(results, indent=2))
    

elif 'gemini' in model_name:
    # 1. Configure Gemini API
    genai.configure(api_key=google_api_key)  # <-- replace with your key
    
    # 2. Load Gemini model
    model = genai.GenerativeModel(model_name)  # or "gemini-1.5-pro"
    

    # 5. Run inference on each pair
    results = []

    for abstract_group in tqdm(pairs):
        abstract_prompts = [f"{a}:{dict_abstracts[a]}" for a in abstract_group]
        abstract_prompt = "\n\n".join(abstract_prompts)
        
        prompt = f"""
        {abstract_prompt}
        
        {task_prompt}
        """
        # print(prompt)
        response = model.generate_content(prompt)
    
        # Try parsing JSON
        text = response.text.strip()
        try:
            parsed = json.loads(text[text.find("{"): text.rfind("}")+1])
        except Exception:
            parsed = {"raw_output": text}
        results.append(parsed)
    
    # 6. Print nicely
    print(json.dumps(results, indent=2))

100%|██████████| 1/1 [00:07<00:00,  7.82s/it]

[
  {
    "raw_output": "\nQuestion: Can you find structures of Mp1p's ligand binding domain 2 (LBD2)?\nReference: \"To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A.\"\n\nQuestion: Can you give me structures of Mp1p's LBD2 showing a dimer interface?\nReference: \"The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity...\"\n\nQuestion: Can you show me structures of Mp1p's LBD2 with a palmitic acid molecule?\nReference: \"electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules.\"\n\nQuestion: Can you find structures of Mp1p's LBD2 with mutations S313A and S332A?\nReference: \"Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity.\"\n\nQuestion: Can you provide structures of Mp1p's LBD2 

In [28]:
# Strip the 'assistant'/'Assistant' prefix from raw Qwen/LLaMA outputs.
if model_type == 'qwen' or model_type == 'llama':
    for i in range(len(results)):
        results[i]['raw_output'] = results[i]['raw_output'].split('assistant')[-1]
        results[i]['raw_output'] = results[i]['raw_output'].split('Assistant')[-1]
        

In [29]:
# Print all raw LLM outputs for manual inspection.
for i, result in enumerate(results):
    # print(result['pairs'])
    print(i)
    print(result['raw_output'])
    print("______________________________________________") #[0]['question'])
    print()

0

Question: Can you find structures of Mp1p's ligand binding domain 2 (LBD2)?
Reference: "To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A."

Question: Can you give me structures of Mp1p's LBD2 showing a dimer interface?
Reference: "The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity..."

Question: Can you show me structures of Mp1p's LBD2 with a palmitic acid molecule?
Reference: "electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules."

Question: Can you find structures of Mp1p's LBD2 with mutations S313A and S332A?
Reference: "Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity."

Question: Can you provide structures of Mp1p's LBD2 showing monomeric and dimeric forms?
Referen

In [30]:
# Strip the 'assistant'/'Assistant' prefix from Qwen/LLaMA final results.
if model_type == 'qwen' or model_type == 'llama':
    for i in range(len(results_final)):
        results_final[i]['raw_output'] = results_final[i]['raw_output'].split('assistant')[-1]
        results_final[i]['raw_output'] = results_final[i]['raw_output'].split('Assistant')[-1]
        

In [31]:
# Print all raw LLM final outputs for manual inspection.

for i, result_final in enumerate(results_final):
    # print(result['pairs'])
    print(i)
    print(result_final['raw_output'])
    print("______________________________________________") #[0]['question'])
    print()

0

Question: Can you find structures of Mp1p's ligand binding domain 2 (LBD2)?

Reference: 
- "To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A."
- "The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity..."
- "electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules."
- "Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity."
- "Analytical ultracentrifugation assays demonstrated that both LBD2 and Mp1p are mostly monomeric in vitro, no matter with or without ligand, and our dimeric crystal structure of LBD2 might be the result of crystal packing."
______________________________________________

1

Question: Can you find structures of cyclic nucleotide-gated (CNG) or hyperpolarization-a

In [32]:
# Print all initial raw outputs for comparison.

for i, result in enumerate(results):
    # print(result['pairs'])
    print(i)
    print(result['raw_output'])
    print("______________________________________________") #[0]['question'])
    print()
#print(results[0]['answer']

0

Question: Can you find structures of Mp1p's ligand binding domain 2 (LBD2)?
Reference: "To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A."

Question: Can you give me structures of Mp1p's LBD2 showing a dimer interface?
Reference: "The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity..."

Question: Can you show me structures of Mp1p's LBD2 with a palmitic acid molecule?
Reference: "electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules."

Question: Can you find structures of Mp1p's LBD2 with mutations S313A and S332A?
Reference: "Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity."

Question: Can you provide structures of Mp1p's LBD2 showing monomeric and dimeric forms?
Referen

In [33]:
# Replace the initial results list with the finalized/cleaned results.
results = results_final

In [34]:
# Display all results.
results

[{'raw_output': '\nQuestion: Can you find structures of Mp1p\'s ligand binding domain 2 (LBD2)?\n\nReference: \n- "To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A."\n- "The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity..."\n- "electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules."\n- "Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity."\n- "Analytical ultracentrifugation assays demonstrated that both LBD2 and Mp1p are mostly monomeric in vitro, no matter with or without ligand, and our dimeric crystal structure of LBD2 might be the result of crystal packing."'},
 {'raw_output': '\nQuestion: Can you find structures of cyclic nucleotide-gated (CNG) or hyperpolarization-activa

In [35]:
# Parse raw LLM outputs into structured question/reference dicts, filtering malformed entries.
import re

new_res = []
new_order = []
for i, result in enumerate(results):
    
    if 'PDB IDs' in result['raw_output'] or 'umber:' in result['raw_output']:
        continue
    else:
        if 'Reference' in result['raw_output']:
            reference = result['raw_output'].split('Reference:')[-1]
        else:
            reference = result['raw_output'].split('Reference:')[-1]
        
        # Matches (Abstract 3), (Abstract3), (Abstract 4), (Abstract4)
        pattern = r'\(Abstract\s*([0-9]+)\)'
        matches = re.findall(pattern, reference)
        matches = list(set(matches))
        
        # if len(matches)>0:
        question = result['raw_output'].split('Reference')[0]
        # answer = result['raw_output'].split('Answer:')[-1].split('Reference')[0]

        question = re.sub(r'\*', '', question)
        # answer = re.sub(r'\*', '', answer)
        reference = re.sub(r'\*', '', reference)

        question = re.sub(r"'", '', question)
        # answer = re.sub(r"'", '', answer)
        reference = re.sub(r"'", '', reference)

        if 'Question:' in question:
            question = question.split('Question:')[-1]
        elif 'Questions:' in question:
            question = question.split('Questions:')[-1]
        elif 'question:' in question:
            question = question.split('question:')[-1]
        elif 'questions:' in question:
            question = question.split('questions:')[-1]
            
        try:
            tmp = {'Pair': result['pairs'], 'Question': question, 'Answer':answer, 'Reference':reference}
        except:
            tmp = {'Question':question, 'Reference':reference}
            # tmp = {'Question':question, 'Answer':answer, 'Reference':reference}
        
        new_res.append(tmp)
        new_order.append(pid_order[i])
        # else:
        #     print(result['raw_output'])
        #     continue


In [36]:
# Display the PubMed ID ordering list.
pid_order

[20053994, 28396445, 22745123, 10023771, 22831795]

In [37]:
# Display the parsed output ordering list.
new_order

[20053994, 28396445, 22745123, 10023771, 22831795]

In [38]:
# Print the number of successfully parsed result entries.
print(len(new_res))

5


In [39]:
# Inspect the first parsed result entry.
new_res[0]

{'Question': ' Can you find structures of Mp1ps ligand binding domain 2 (LBD2)?\n\n',
 'Reference': ' \n- "To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A."\n- "The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity..."\n- "electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules."\n- "Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity."\n- "Analytical ultracentrifugation assays demonstrated that both LBD2 and Mp1p are mostly monomeric in vitro, no matter with or without ligand, and our dimeric crystal structure of LBD2 might be the result of crystal packing."'}

In [40]:
# Back up the parsed results list before in-place mutation.
import copy

# save = copy.deepcopy(new_res)
# new_res = save

In [41]:
# Resolve PubMed IDs to RCSB PDB IDs and insert them into each result entry.

for index in range(len(new_order)):

    if len(new_res[index]) == 0:
        continue

    # original_rid = new_order[index]
    # print(original_rid)
    original_pid = new_order[index]#[dict_rid_info[original_rid]['pubmed_id']]
    linked_rid = dict_pid_rid[original_pid]
    # all_rid = [item for sublist in linked_rid for item in sublist]



        # Replacement function
    def replace_with_pid(reference, pid):
        # pid = dict_rid_info[original_rid]['pubmed_id']
        webpage = 'https://pubmed.ncbi.nlm.nih.gov/' + str(pid) + '/'
        return f'{reference}\nSource: {webpage}'  # 👈 Replace "Abstract x" with PID only
    
    # Perform substitution
    # new_references = re.sub(pattern, replace_with_pid, references)
    if selected != 'multiple':
        new_res[index]['Pubmed_IDs'] = str(original_pid) + f' (https://pubmed.ncbi.nlm.nih.gov/{original_pid}/)'
    else:
        new_res[index]['Pubmed_IDs'] = str(original_pid)
        
    if selected != 'multiple':    
        linked_rid = [f'{i} (https://www.rcsb.org/structure/{i}/)' for i in linked_rid]
    new_res[index]['RCSB_PDB_IDs'] = (', ').join(linked_rid)
    
    references = new_res[index]['Reference']
    new_res[index]['Reference'] = replace_with_pid(references, original_pid)

print('Finished')

Finished


In [42]:
# Print the total count of parsed result entries.
print(len(new_res))

5


In [43]:
# Inspect the second parsed result entry.
print(new_res[1])

{'Question': ' Can you find structures of cyclic nucleotide-gated (CNG) or hyperpolarization-activated cyclic nucleotide-regulated (HCN) ion channels?\n\n', 'Reference': ' CNG and HCN ion channels play crucial physiological roles in phototransduction, olfaction, and cardiac pace making.\nSource: https://pubmed.ncbi.nlm.nih.gov/28396445/', 'Pubmed_IDs': '28396445 (https://pubmed.ncbi.nlm.nih.gov/28396445/)', 'RCSB_PDB_IDs': '5V4S (https://www.rcsb.org/structure/5V4S/)'}


In [44]:
# Find and print all result entries whose question contains the word 'hot'.
for items in new_res:
    if 'hot' in items['Question']:
        print(items['Question'])
        print(items['Reference'])
        print()


In [45]:
# Count and print entries with empty or newline-only question fields.
c = 0 
for i, data in enumerate(new_res):
    # if data['Pubmed_IDs'] in pid:
    if data['Question'] == '' or data['Question'] == '\n':
        print(i)
        text = (
            f"[Sample {c+1}]\n\n"
            f"Question: {data['Question'].strip()}\n\n"
            # f"Answer: {data['Answer'].strip()}\n"
            f"Reference: \n{data['Reference'].strip()}\n\n"
            f"PubMed IDs: {data['Pubmed_IDs']}\n"
            f"PDB IDs: {data['RCSB_PDB_IDs']}"
        )
        print(text)
        print()
        print()
        c += 1


In [46]:
# Inspect the last result entry.
new_res[-1]

{'Question': ' Can you find structures of GH19 chitinases with known substrate-binding modes?\n\n',
 'Reference': ' \n- "These are the first crystallographic data showing the oligosaccharide-binding mode of a family GH19 chitinase."\nSource: https://pubmed.ncbi.nlm.nih.gov/22831795/',
 'Pubmed_IDs': '22831795 (https://pubmed.ncbi.nlm.nih.gov/22831795/)',
 'RCSB_PDB_IDs': '4DWX (https://www.rcsb.org/structure/4DWX/)'}

In [47]:
# Identify and print indices of entries with empty or whitespace-only question fields.
for idx,item in enumerate(new_res):
    if new_res[idx]['Question'] == '\n':
        print('empty')
        print(idx)
        print(new_res[idx]['Question'])
        
    if new_res[idx]['Question'] == '':
        print('empty')
        print(idx)
        print(new_res[idx]['Question'])

    if "selected question" not in new_res[idx]['Question']:
        new_res[idx]['Question'] = item['Question'].replace("\n", "")
        new_res[idx]['Question'] = item['Question'].replace("userCan", "Can")
        new_res[idx]['Question'] = item['Question'].replace(": Can", "Can")
        # new_res[idx]['Question'] = item['Question'].replace(" Can", "Can")
    else:
        print('select question in question')
        print(idx)
        print(new_res[idx]['Question'])


In [48]:
# Remove all entries with empty or whitespace-only question fields.
new_res = [item for item in new_res if item["Question"].strip() != ""]
new_res = [item for item in new_res if "selected question" not in item["Question"].strip()]

In [49]:
# Print all questions with their indices for final review.
for idx,item in enumerate(new_res):
    print(idx)
    print(item['Question'])

0
 Can you find structures of Mp1ps ligand binding domain 2 (LBD2)?
1
 Can you find structures of cyclic nucleotide-gated (CNG) or hyperpolarization-activated cyclic nucleotide-regulated (HCN) ion channels?
2
 Can you find structures of Mesorhizobium sp. strain LUK S-selective transaminase (MesAT) in complex with various amino acids?
3
 Can you find structures of TCR-MHC glycopeptide complexes?
4
 Can you find structures of GH19 chitinases with known substrate-binding modes?


In [50]:
# Collect all used categories, PubMed IDs, and PDB IDs from the final results.

used = sorted_category
used_pid = new_order
used_pdb = []
for item in new_res:
    temp = item['RCSB_PDB_IDs'].split(', ')
    used_pdb += temp


In [51]:
# print entries.
c = 0 
for i, data in enumerate(new_res):
    # if data['Pubmed_IDs'] in pid:
    print(i)
    text = (
        f"[Sample {c+1}]\n\n"
        f"Question: {data['Question'].strip()}\n\n"
        # f"Answer: {data['Answer'].strip()}\n"
        f"Reference: \n{data['Reference'].strip()}\n\n"
        f"PubMed IDs: {data['Pubmed_IDs']}\n"
        f"PDB IDs: {data['RCSB_PDB_IDs']}"
    )
    print(text)
    print()
    print()
    c += 1


0
[Sample 1]

Question: Can you find structures of Mp1ps ligand binding domain 2 (LBD2)?

Reference: 
- "To understand the function of Mp1p, we have determined the x-ray crystal structure of its ligand binding domain 2 (LBD2) to 1.3 A."
- "The structure reveals a dimer between the two molecules. The dimer interface forms a ligand binding cavity..."
- "electron density was observed for a palmitic acid molecule interacting with LBD2 indirectly through hydrogen bonding networks via two structural water molecules."
- "Mutations of ligand-binding residues, namely S313A and S332A, resulted in a 9-fold suppression of ligand binding affinity."
- "Analytical ultracentrifugation assays demonstrated that both LBD2 and Mp1p are mostly monomeric in vitro, no matter with or without ligand, and our dimeric crystal structure of LBD2 might be the result of crystal packing."
Source: https://pubmed.ncbi.nlm.nih.gov/20053994/

PubMed IDs: 20053994 (https://pubmed.ncbi.nlm.nih.gov/20053994/)
PDB IDs: 3L1N 

In [52]:
# Print the total counts of used categories, PubMed IDs, and PDB IDs.
print(len(used))
print(len(used_pid))
print(len(used_pdb))

7
5
5


In [53]:
# Save used category/PID/PDB tracking info to a JSON file for future deduplication.
if selected == 'single':
    data = {
        "used_category_name_single": used,
        "used_pid_single": used_pid,
        "used_pdb_id_single":used_pdb
    }
    
    path = OUTPUT_DIR / 'used_info_single.json'
    with open(path, "w") as f:
        json.dump(data, f, indent=4)
else:
    data = {
        "used_category_name_multiple": used,
        "used_pid_multiple": used_pid,
        "used_pdb_id_multiple": used_pdb
    }
    
    path = OUTPUT_DIR / 'used_info_multiple.json'
    with open(path, "w") as f:
        json.dump(data, f, indent=4)

In [54]:
# Save the final cleaned QA results to a JSON file.
# Save as JSON file

# name = category_query_name
# name = re.sub(r" ", '_', name)
# rids = ("_").join(rid_order)

abs_type = selected
file_name = f'qwen_{len(new_res)}_{abs_type}_abstract'#f"abstract_pairs_{name}_{rids}"
print(file_name)

path = OUTPUT_DIR
with open(f"{path}/{file_name}_03102026.json", "w") as f:
    json.dump(new_res, f)

print(f"JSON file saved as {file_name}.json")


qwen_5_single_abstract
JSON file saved as qwen_5_single_abstract.json


In [55]:
# (Empty placeholder cell.)


In [56]:
# (Empty placeholder cell.)
